# Genned: adversarial fine-tuning (Phase 2b, issue #15), attempt 3

> **Open a fresh copy of this notebook from GitHub** (Colab: *File → Open notebook → GitHub → as791/genned →
> notebooks/adversarial_finetune.ipynb*). Don't re-run an old tab: attempt 2 accidentally ran attempt 1's cached
> cells. The setup cell below refuses to run if the code isn't the attempt-3 version.

Fine-tunes the app's detectors so that invisible perturbations up to **4/255** can't flip them.

**History.** Attempt 1 (PGD-AT, lr 1e-5) learned no robustness. Attempt 2 (TRADES β 6, lr 1e-4) *collapsed*: the
model learned to answer 50/50 for every image. Attempt 3 is conservative:
- learning rate 2e-5;
- the robustness weight ramps 0 → 3 over 2 epochs, and the attack strength ramps 0 → 4/255 over 1 epoch;
- validation every half epoch, and training **stops automatically** if clean accuracy starts collapsing;
- a checkpoint is saved and uploaded **only** if clean accuracy held (AUC within 0.01 of the start).

**Held-out data:** training uses **Defactify's train split only**. MJ/DALL·E/SD/NBP and the two video sets stay unseen.

**Before you start**
1. *Runtime → Change runtime type → GPU* (Colab), or *Accelerator → GPU* + *Internet on* (Kaggle).
2. Secret **`HF_TOKEN`** (write scope; account has accepted the gated `Dafilab/ai-image-detector` terms).
3. *Run all.* Community Forensics: about **45–60 min** on a T4. Watch the *validation* lines (every half epoch):
   clean AUC should stay near the start while *robust @4/255* rises.

**What leaves this machine:** only weights + a JSON of numbers, to your **private** HF repo as `*-robust-v3.pt`.

In [ ]:
!nvidia-smi || echo 'No GPU: switch the runtime to a GPU before continuing.'

In [ ]:
import os, subprocess, shutil

def secret(name):
    try:
        from google.colab import userdata  # Colab
        return userdata.get(name)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient  # Kaggle
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)

os.environ["HF_TOKEN"] = secret("HF_TOKEN") or ""
assert os.environ["HF_TOKEN"], "Add an HF_TOKEN secret (write scope) first - see the first cell."
gh = secret("GITHUB_TOKEN")  # only needed if the repo is private
url = f"https://{gh}@github.com/as791/genned.git" if gh else "https://github.com/as791/genned.git"
os.chdir("/content" if os.path.isdir("/content") else os.getcwd())
if os.path.isdir("genned"):
    shutil.rmtree("genned")  # always start from the latest code
subprocess.run(["git", "clone", "--depth", "1", url, "genned"], check=True)
os.chdir("genned")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
assert "adv-finetune-v3" in open("tools/adv_finetune.py").read(), "Old code: re-open this notebook from GitHub."
print("Attempt-3 code confirmed.")

In [ ]:
!pip install -q "timm>=1.0,<2" "huggingface_hub>=0.23,<1" "datasets>=2.19,<5" "onnxruntime>=1.18,<2" "pillow>=10,<12"
from huggingface_hub import HfApi
HF_REPO = HfApi().whoami()["name"] + "/genned-robust"
print("Weights will be uploaded (privately) to", HF_REPO)

## Training data: Defactify train split only; validation: Defactify validation split
`--strict-split` makes the download fail rather than silently fall back to the test split the benchmarks use.

In [ ]:
PER_CLASS = 3000  # images per class; lower it if the session is short on time or disk

!python tools/fetch_eval_data.py --dataset Rajarshi-Roy-research/Defactify_Image_Dataset \
    --split train --strict-split --out data/train/defactify --per-class {PER_CLASS} --max-scan 25000 \
    --label-col Label_A --ai-values 1 --real-values 0 \
    --generator-col Label_B --generator-names real,sd21,sdxl,sd3,dalle3,midjourney6
!python tools/fetch_eval_data.py --dataset Rajarshi-Roy-research/Defactify_Image_Dataset \
    --split validation --strict-split --out data/val/defactify --per-class 150 \
    --label-col Label_A --ai-values 1 --real-values 0 \
    --generator-col Label_B --generator-names real,sd21,sdxl,sd3,dalle3,midjourney6

## 1. Community Forensics ViT-S 224 (~45–60 min on a T4)
All recipe settings come from the script's defaults (no hard-coded epochs or learning rate here).

In [ ]:
!python tools/adv_finetune.py --model commfor \
    --train-data data/train/defactify --val-data data/val/defactify \
    --per-class {PER_CLASS} --tag v3 --out runs --push-to-hub {HF_REPO}

## 2. Dafilab EfficientNet-B4 (only if step 1 is usable; 2–3 h on a T4)
Runs only if Community Forensics kept its clean accuracy **and** reached ≥ 30% robust accuracy at 4/255.

In [ ]:
import json
log = json.load(open("runs/commfor-train-log-v3.json"))
sel = log["selected"]
usable = log["meets_clean_gate"] and sel is not None and sel["robust_acc_4"] >= 0.30
print("Community Forensics:", "stopped - " + log["stopped"] if log["stopped"] else "ran to the end")
if sel:
    print(f"  selected @ epoch {sel['epoch']}: clean AUC {sel['clean_auc']:.3f} (start {log['base']['clean_auc']:.3f}), "
          f"robust @4/255 {sel['robust_acc_4']:.1%} (start {log['base']['robust_acc_4']:.1%})")
if usable:
    !python tools/adv_finetune.py --model bundled \
        --train-data data/train/defactify --val-data data/val/defactify \
        --per-class {PER_CLASS} --tag v3 --out runs --push-to-hub {HF_REPO}
else:
    print("Not usable (clean gate or < 30% robust): skipping EfficientNet-B4.")

In [ ]:
import glob
for path in sorted(glob.glob("runs/*-train-log-v3.json")):
    log = json.load(open(path))
    print(f"{log['model']}: clean gate met: {log['meets_clean_gate']}, stopped: {log['stopped']}")
    print(f"  start: clean AUC {log['base']['clean_auc']:.3f}, robust @2/255 {log['base']['robust_acc_2']:.1%}, "
          f"@4/255 {log['base']['robust_acc_4']:.1%}")
    for e in log["evaluations"]:
        print(f"  epoch {e['epoch']}: eps {e['eps_255']}/255 beta {e['beta']}: clean AUC {e['clean_auc']:.3f}, "
              f"acc {e['clean_acc']:.1%}, robust @2/255 {e['robust_acc_2']:.1%}, @4/255 {e['robust_acc_4']:.1%}"
              f"{'' if e['eligible'] else '  (not eligible)'}")
print()
print("Done. Paste this summary to Claude.")